# 01 Silver - Clean and conform

**Bronze → Silver.** Reads the raw workshop data in **FabricWorkshopDemo** (read only) and writes clean tables to
**AdventureWorksSilver**, schema `dbo`:

| Silver table | Grain | Built from |
|---|---|---|
| `online_sales_line` | one row per online order line | `sales.csv` + yearly files, deduplicated |
| `reseller_order_line` | one row per reseller order line | `Tables/dbo/orders` |
| `product` | one row per product | `products.csv`, name parsed into model / variant / color |
| `customer` | one row per customer, both channels | derived from the two order tables |

**Practices used here**
- Bronze is never modified; Silver can always be rebuilt from it.
- Explicit types (dates, integers, `decimal(19,4)` money), trimmed text, lower-case e-mails.
- One deduplication rule, written down and counted.
- Data quality checks (`assert`) stop the run before bad data is saved.
- Each table is written with **overwrite**: re-running gives the same result (idempotent).
- `_source_file` keeps the lineage of every online line back to its Bronze file.

Run cell by cell, or **Run all** (about 2 minutes after Spark starts). The findings behind these rules are in
notebook **00 Bronze - Explore the raw data**.


In [ ]:
# Configuration and helpers
from pyspark.sql import functions as F, Window as W

def lakehouse_path(name):
    """OneLake path of a lakehouse in this workspace, found by name."""
    return notebookutils.lakehouse.get(name)["properties"]["abfsPath"]

BRONZE = lakehouse_path("FabricWorkshopDemo")  # read only
SALES_COLS = ["SalesOrderNumber", "SalesOrderLineNumber", "OrderDate", "CustomerName", "EmailAddress",
              "Item", "Quantity", "UnitPrice", "TaxAmount"]

def bronze_csv(path, header=True):
    """Reads a Bronze CSV as text; files without a header get the sales.csv column names."""
    df = spark.read.option("header", header).option("quote", '"').option("escape", '"').csv(f"{BRONZE}/Files/{path}")
    return (df if header else df.toDF(*SALES_COLS)).withColumn("_source_file", F.lit(path))

def save(df, name):
    """Writes a Silver table in the default lakehouse (AdventureWorksSilver), replacing its previous content."""
    df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"dbo.{name}")
    print(f"saved dbo.{name}: {spark.table(f'dbo.{name}').count():,} rows")

## 1. Online order lines
### 1a. Read every file that holds online lines

In [ ]:
raw = bronze_csv("data/sales.csv")
for year in ("2019", "2020", "2021"):
    raw = raw.unionByName(bronze_csv(f"bronze/{year}.csv", header=False))
display(raw.groupBy("_source_file").count().orderBy("_source_file"))

### 1b. Deduplicate and type
**Rule:** one row per `SalesOrderNumber` + `SalesOrderLineNumber`. When the line is in several files, keep the
one from `sales.csv`. Then trim text, lower-case the e-mail and cast every column to its type.

In [ ]:
first = W.partitionBy("SalesOrderNumber", "SalesOrderLineNumber").orderBy(
    F.when(F.col("_source_file") == "data/sales.csv", 0).otherwise(1), "_source_file")
online = (raw.withColumn("_rn", F.row_number().over(first)).where("_rn = 1")
             .select(F.trim("SalesOrderNumber").alias("SalesOrderNumber"),
                     F.col("SalesOrderLineNumber").cast("int").alias("LineNumber"),
                     F.to_date("OrderDate").alias("OrderDate"),
                     F.trim("CustomerName").alias("CustomerName"),
                     F.lower(F.trim("EmailAddress")).alias("EmailAddress"),
                     F.trim("Item").alias("ProductName"),
                     F.col("Quantity").cast("int").alias("Quantity"),
                     F.col("UnitPrice").cast("decimal(19,4)").alias("UnitPrice"),
                     F.col("TaxAmount").cast("decimal(19,4)").alias("TaxAmount"),
                     "_source_file"))
print(f"read {raw.count():,} lines, kept {online.count():,}, removed {raw.count() - online.count():,} duplicates")
display(online.limit(5))

### 1c. Quality checks, then save

In [ ]:
assert online.where("SalesOrderNumber IS NULL OR LineNumber IS NULL OR OrderDate IS NULL").count() == 0, "missing key or date"
assert online.where("Quantity <= 0 OR UnitPrice < 0").count() == 0, "bad quantity or price"
assert online.where("EmailAddress IS NULL").count() == 0, "line without customer e-mail"
assert online.select("SalesOrderNumber", "LineNumber").distinct().count() == online.count(), "duplicate order line"
save(online, "online_sales_line")

## 2. Reseller order lines
Same column names as the online lines, so Gold can stack both channels. Order numbers get the `SO` prefix of the
online channel; the workshop's helper column `MonthNo` is dropped (Gold has a full date dimension).

In [ ]:
reseller = (spark.read.format("delta").load(f"{BRONZE}/Tables/dbo/orders")
            .select(F.concat(F.lit("SO"), F.col("SalesOrderID")).alias("SalesOrderNumber"),
                    F.col("LineItem").cast("int").alias("LineNumber"),
                    F.to_date("OrderDate").alias("OrderDate"),
                    F.col("CustomerID").cast("int").alias("ResellerAccountId"),
                    F.col("ProductID").cast("int").alias("ProductId"),
                    F.col("OrderQty").cast("int").alias("Quantity"),
                    F.col("LineItemTotal").cast("decimal(19,4)").alias("LineAmount")))
assert reseller.select("SalesOrderNumber", "LineNumber").distinct().count() == reseller.count(), "duplicate order line"
assert reseller.join(online, "SalesOrderNumber").count() == 0, "reseller order number collides with an online order"
save(reseller, "reseller_order_line")
display(reseller.limit(5))

## 3. Products
The catalog name packs three things: `Mountain-100 Silver, 44` → model **Mountain-100**, color **Silver**,
variant (size) **44**. Each category also gets its **group**: Bikes, Components, Clothing or Accessories.

In [ ]:
GROUPS = {"Bikes": ["Mountain Bikes", "Road Bikes", "Touring Bikes"],
          "Components": ["Bottom Brackets", "Brakes", "Chains", "Cranksets", "Derailleurs", "Forks", "Handlebars",
                         "Headsets", "Mountain Frames", "Pedals", "Road Frames", "Saddles", "Touring Frames", "Wheels"],
          "Clothing": ["Bib-Shorts", "Caps", "Gloves", "Jerseys", "Shorts", "Socks", "Tights", "Vests"],
          "Accessories": ["Bike Racks", "Bike Stands", "Bottles and Cages", "Cleaners", "Fenders", "Helmets",
                          "Hydration Packs", "Lights", "Locks", "Panniers", "Pumps", "Tires and Tubes"]}
group_of = F.create_map(*[F.lit(x) for g, cats in GROUPS.items() for c in cats for x in (c, g)])
COLORS = ["Black", "Blue", "Grey", "Multi", "Red", "Silver", "White", "Yellow"]
name = F.trim("ProductName")
product = (bronze_csv("products/products.csv")
           .select(F.col("ProductID").cast("int").alias("ProductId"), name.alias("ProductName"),
                   F.trim(F.regexp_replace(F.split(name, ",").getItem(0), rf"\s*-?\s*\b({'|'.join(COLORS)})\b", "")).alias("ProductModel"),
                   F.trim(F.element_at(F.split(name, ","), 2)).alias("Variant"),
                   F.coalesce(*[F.when(name.rlike(rf"\b{c}\b"), F.lit(c)) for c in COLORS]).alias("Color"),
                   F.trim("Category").alias("Category"),
                   F.col("ListPrice").cast("decimal(19,4)").alias("ListPrice"))
           .withColumn("CategoryGroup", group_of[F.col("Category")]))
assert product.where("CategoryGroup IS NULL").count() == 0, "category without a group"
assert product.select("ProductId").distinct().count() == product.count(), "duplicate product id"
assert online.join(product, "ProductName", "left_anti").count() == 0, "online line with a product name not in the catalog"
assert reseller.join(product, "ProductId", "left_anti").count() == 0, "reseller line with an unknown product id"
save(product, "product")
display(product.where("ProductName LIKE 'Mountain-100%' OR ProductName LIKE 'HL Road Frame%'").orderBy("ProductName"))

## 4. Customers of both channels
Online customers have no id in the source: the e-mail identifies them. Reseller accounts are identified by their
account id, prefixed `RS-` so the two kinds can never collide.

In [ ]:
online_c = (online.groupBy("EmailAddress")
            .agg(F.first("CustomerName").alias("CustomerName"), F.min("OrderDate").alias("FirstOrderDate"))
            .select(F.col("EmailAddress").alias("CustomerAlternateKey"), "CustomerName", "EmailAddress",
                    F.lit("Individual").alias("CustomerType"), F.lit("Online").alias("AcquisitionChannel"),
                    "FirstOrderDate"))
reseller_c = (reseller.groupBy("ResellerAccountId").agg(F.min("OrderDate").alias("FirstOrderDate"))
              .select(F.concat(F.lit("RS-"), F.col("ResellerAccountId")).alias("CustomerAlternateKey"),
                      F.concat(F.lit("Reseller account "), F.col("ResellerAccountId")).alias("CustomerName"),
                      F.lit(None).cast("string").alias("EmailAddress"),
                      F.lit("Reseller").alias("CustomerType"), F.lit("Reseller").alias("AcquisitionChannel"),
                      "FirstOrderDate"))
customer = online_c.unionByName(reseller_c)
assert customer.select("CustomerAlternateKey").distinct().count() == customer.count(), "duplicate customer"
save(customer, "customer")
display(customer.groupBy("CustomerType").count())

## 5. Bronze vs Silver
Same orders and products, half the online lines: the duplicates are gone. Summing Bronze as it is would double
online revenue, which is exactly the kind of error Silver exists to prevent.

In [ ]:
raw.createOrReplaceTempView("bronze_lines")  # every online line from all four files, duplicates included
bronze_csv("products/products.csv").createOrReplaceTempView("bronze_products")

In [ ]:
%%sql
SELECT 'online lines' AS what, (SELECT COUNT(*) FROM bronze_lines) AS bronze, COUNT(*) AS silver
FROM AdventureWorksSilver.dbo.online_sales_line
UNION ALL
SELECT 'online orders', (SELECT COUNT(DISTINCT SalesOrderNumber) FROM bronze_lines), COUNT(DISTINCT SalesOrderNumber)
FROM AdventureWorksSilver.dbo.online_sales_line
UNION ALL
SELECT 'online revenue', (SELECT ROUND(SUM(CAST(UnitPrice AS DECIMAL(19,4)) * CAST(Quantity AS INT)), 2) FROM bronze_lines),
       ROUND(SUM(UnitPrice * Quantity), 2)
FROM AdventureWorksSilver.dbo.online_sales_line
UNION ALL
SELECT 'reseller lines', (SELECT COUNT(*) FROM FabricWorkshopDemo.dbo.orders), COUNT(*)
FROM AdventureWorksSilver.dbo.reseller_order_line
UNION ALL
SELECT 'products', (SELECT COUNT(*) FROM bronze_products), COUNT(*) FROM AdventureWorksSilver.dbo.product

Next: **02 Gold - Sales star schema**.